# Deep Learning Audio Beat Tracking — TCN

A 21,800-parameter Temporal Convolutional Network in PyTorch for beat tracking
on the Ballroom dataset. Reproduces and extends the architecture from
*Davies & Böck (2019), Temporal Convolutional Networks for Musical Audio Beat
Tracking.*

**Pipeline:** 81-band log-CQT spectrogram → 2D conv front-end → 1D dilated TCN
→ per-frame beat activation → dynamic-programming decoding (`librosa.beat.beat_track`).

**Headline results** (20% held-out test split, 698 Ballroom tracks):

| Metric | Score |
|---|---|
| F-measure | **0.831** |
| AMLt      | **0.923** |

Class imbalance (>95% non-beat frames) is handled with a weighted BCE loss
(`pos_weight=30`) and Gaussian-widened beat targets.

---

## 0. Imports

`mirdata` provides convenient access to the Ballroom dataset, including its
official beat annotations. `mir_eval` provides the standard beat-tracking
metrics (F-measure, CMLc/CMLt, AMLc/AMLt, Information Gain).

In [ ]:
import os
import random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

import librosa
import librosa.display
import mir_eval
import mirdata

from scipy.signal import medfilt
from scipy.ndimage import maximum_filter1d
import matplotlib.pyplot as plt

## 1. Data Preparation & Feature Extraction

We use a **log-magnitude Constant-Q Transform** (CQT) with 81 bins
(7 octaves × 12 bins/octave + a few extra for low frequencies), `fmin=30 Hz`,
and a 10 ms hop. The CQT's logarithmic frequency spacing matches musical
perception better than a linear STFT and gives the front-end conv layers a
more structured input.

Beat targets are widened: each ground-truth beat frame is set to 1.0, and the
two adjacent frames either side are set to 0.5. This soft target stabilises
training (a perfectly hard target makes the loss unforgiving of 1-frame timing
errors).

In [ ]:
def compute_log_cqt(file_path):
    """Generate the 81-band log-magnitude CQT spectrogram for an audio file."""
    sr = 44100
    hop_length = int(sr * 0.01)  # 10 ms
    y, _ = librosa.load(file_path, sr=sr)

    cqt = librosa.cqt(
        y, sr=sr, hop_length=hop_length,
        fmin=30, n_bins=81, bins_per_octave=12,
        filter_scale=1.0,
    )

    log_cqt = librosa.amplitude_to_db(np.abs(cqt), ref=np.max)
    log_cqt_tensor = torch.tensor(log_cqt, dtype=torch.float32).unsqueeze(0).unsqueeze(0)
    return log_cqt_tensor


def create_target_array(beat_times, num_frames, sr=44100, hop_length=441):
    """Convert beat times (seconds) into a frame-level target array.

    Beat frames are set to 1.0; the two neighbouring frames are set to 0.5
    (a Gaussian-style widening). This avoids penalising 1-frame off-by-one
    predictions during training.
    """
    target = np.zeros(num_frames)
    if beat_times is None or len(beat_times) == 0:
        return target

    beat_frames = np.round((beat_times * sr) / hop_length).astype(int)

    for frame in beat_frames:
        if 0 <= frame < num_frames:
            target[frame] = 1.0
            for offset in [-2, -1, 1, 2]:
                adj = frame + offset
                if 0 <= adj < num_frames:
                    target[adj] = max(target[adj], 0.5)
    return target

### Pre-computation pipeline

Computing the CQT on-the-fly during training is the bottleneck — `librosa.cqt`
is CPU-bound and takes ~3–5 seconds per track. With 698 tracks × 50 epochs that
would dominate training time. Instead, we compute features once, save them as
tensor `.pt` files, and load directly in the `Dataset.__getitem__`. This
eliminates the CPU bottleneck and lets the GPU stay busy during training.

In [ ]:
def precompute_dataset(ballroom_data, save_dir="./precomputed_data"):
    """Cache CQT features and beat targets for every track in the dataset."""
    os.makedirs(save_dir, exist_ok=True)
    print(f"Pre-computing features for {len(ballroom_data.track_ids)} files. "
          f"This will take ~30-40 mins on CPU...")

    for i, track_id in enumerate(ballroom_data.track_ids):
        feature_path = os.path.join(save_dir, f"{track_id}_feature.pt")
        target_path = os.path.join(save_dir, f"{track_id}_target.pt")

        if os.path.exists(feature_path) and os.path.exists(target_path):
            continue

        try:
            track = ballroom_data.track(track_id)
            features = compute_log_cqt(track.audio_path)[0]
            num_frames = features.shape[-1]

            beat_times = track.beats.times if track.beats is not None else np.array([])
            targets = create_target_array(beat_times, num_frames)

            torch.save(features, feature_path)
            torch.save(torch.tensor(targets, dtype=torch.float32), target_path)

            if (i + 1) % 50 == 0:
                print(f"  Processed {i + 1} / {len(ballroom_data.track_ids)} files...")
        except Exception as e:
            print(f"  Error processing {track_id}: {e}")

    print("Pre-computation complete.")


class PrecomputedBallroomDataset(Dataset):
    """Loads cached CQT tensors from disk — no CPU work during training."""

    def __init__(self, track_ids, data_dir="./precomputed_data"):
        self.track_ids = track_ids
        self.data_dir = data_dir

    def __len__(self):
        return len(self.track_ids)

    def __getitem__(self, idx):
        track_id = self.track_ids[idx]
        features = torch.load(
            os.path.join(self.data_dir, f"{track_id}_feature.pt"),
            weights_only=True,
        )
        targets = torch.load(
            os.path.join(self.data_dir, f"{track_id}_target.pt"),
            weights_only=True,
        )
        return features, targets

## 2. Network Architecture: Temporal Convolutional Network

The model has two stages:

1. **2D conv front-end** that operates jointly on time × frequency, ending in
   a vertical pooling stack that collapses the frequency axis to a single row.
2. **1D dilated TCN** (Bai et al. 2018) that captures long-range temporal
   dependencies. Dilations of `[1, 2, 4, 8]` give a receptive field large
   enough to span several beats at typical tempi.

Each TCN block is wrapped in a residual connection. Total parameters: ~21,800.

In [ ]:
class TCNBeatTracker(nn.Module):
    """Temporal Convolutional Network for per-frame beat activation."""

    def __init__(self):
        super().__init__()

        # Front-end: 2D conv over (frequency, time)
        self.conv1 = nn.Conv2d(1, 16, kernel_size=(3, 3), padding=(0, 1))
        self.pool1 = nn.MaxPool2d(kernel_size=(3, 1))
        self.conv2 = nn.Conv2d(16, 16, kernel_size=(3, 3), padding=(0, 1))
        self.pool2 = nn.MaxPool2d(kernel_size=(3, 1))
        self.conv3 = nn.Conv2d(16, 16, kernel_size=(8, 1), padding=(0, 0))
        self.dropout_conv = nn.Dropout(0.1)

        # 1D dilated TCN with residual connections
        self.tcn_layers = nn.ModuleList([
            nn.Conv1d(16, 16, kernel_size=5, dilation=d, padding="same")
            for d in [1, 2, 4, 8]
        ])
        self.dropout_tcn = nn.Dropout(0.1)

        self.fc_out = nn.Linear(16, 1)

    def forward(self, x):
        x = F.elu(self.conv1(x))
        x = self.pool1(x)
        x = F.elu(self.conv2(x))
        x = self.pool2(x)
        x = F.elu(self.conv3(x))
        x = self.dropout_conv(x)
        x = x.squeeze(2)  # collapse the frequency axis

        for tcn_layer in self.tcn_layers:
            residual = x
            x = F.elu(tcn_layer(x))
            x = self.dropout_tcn(x)
            x = x + residual

        x = x.transpose(1, 2)
        return self.fc_out(x).squeeze(-1)

## 3. Training

**Loss.** Beat frames are <5% of all frames, so a vanilla BCE would let the
model trivially predict "no beat" everywhere. We use
`BCEWithLogitsLoss(pos_weight=30)`, which scales the positive-class loss term
by 30× — empirically the cleanest fix for the imbalance and matching the ratio
of negative to positive frames.

**Optimiser.** Adam with `lr=1e-3`, batch size 1 (variable-length sequences).
50 epochs.

In [ ]:
def train_model(model, train_loader, num_epochs=50):
    """Train the TCN with weighted BCE on a 1-track-per-batch loader."""
    model.train()
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device)
    print(f"Training on: {device}")

    pos_weight = torch.tensor([30.0]).to(device)
    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
    optimizer = optim.Adam(model.parameters(), lr=0.001)

    for epoch in range(num_epochs):
        running_loss = 0.0
        for features, targets in train_loader:
            features, targets = features.to(device), targets.to(device)
            optimizer.zero_grad()
            outputs = model(features)
            loss = criterion(outputs, targets)
            loss.backward()
            optimizer.step()
            running_loss += loss.item()

        print(f"--- Epoch {epoch+1}/{num_epochs}  Avg Loss: {running_loss/len(train_loader):.4f} ---")

    return model

## 4. Inference: `beatTracker(inputFile)`

This is the entry point required by the coursework spec. Given an audio file
path, it returns `(beats, downbeats)` arrays of times in seconds.

The TCN outputs a per-frame probability of being a beat. We then run
`librosa.beat.beat_track` on the activation, which uses dynamic programming
to find a globally consistent tempo and beat sequence.

Downbeat estimation is left as `[]` — out of scope for this assignment.

In [ ]:
def beatTracker(inputFile):
    """Return (beat_times, downbeat_times) for an audio file."""
    features = compute_log_cqt(inputFile)

    model = TCNBeatTracker()
    try:
        script_dir = os.path.dirname(os.path.abspath(__file__))
        model_path = os.path.join(script_dir, "tcn_beat_tracker.pth")
    except NameError:
        model_path = "tcn_beat_tracker.pth"

    try:
        model.load_state_dict(
            torch.load(model_path, map_location=torch.device("cpu"), weights_only=True)
        )
    except FileNotFoundError:
        print(f"Warning: Model weights not found at {model_path}. Returning empty predictions.")
        return [], []

    model.eval()
    with torch.no_grad():
        logits = model(features).squeeze()
        beat_activation = torch.sigmoid(logits).cpu().numpy()

    sr = 44100
    hop_length = int(sr * 0.01)
    _, beats_frames = librosa.beat.beat_track(
        onset_envelope=beat_activation,
        sr=sr,
        hop_length=hop_length,
        tightness=100,
    )

    beats = librosa.frames_to_time(beats_frames, sr=sr, hop_length=hop_length)
    downbeats = []  # downbeat detection is out of scope
    return beats, downbeats

## 5. Evaluation

We use the standard `mir_eval.beat.evaluate` suite:

- **F-measure** — strict ±70 ms tolerance.
- **CMLc / CMLt** — Correct Metric Level (Continuous / Total): rewards beats at
  the *exact* metrical level of the ground truth.
- **AMLc / AMLt** — Allowed Metric Level: also accepts double/half-tempo
  predictions, which are perceptually valid.
- **Information Gain** — entropy-based measure of beat consistency.

In [ ]:
def evaluate_dataset(test_ids, ballroom_data):
    """Run `beatTracker` on every test track and report mir_eval averages."""
    total_metrics = {
        "F-measure": 0.0, "CMLc": 0.0, "CMLt": 0.0,
        "AMLc": 0.0, "AMLt": 0.0, "Information gain": 0.0,
    }
    valid_files = 0

    print(f"\n--- Evaluating {len(test_ids)} test files ---")
    for track_id in test_ids:
        track = ballroom_data.track(track_id)
        if track.beats is None or len(track.beats.times) == 0:
            continue

        ref_beats = track.beats.times
        est_beats, _ = beatTracker(track.audio_path)
        est_beats = np.array(est_beats)

        if len(est_beats) > 0:
            metrics = mir_eval.beat.evaluate(ref_beats, est_beats)
            total_metrics["F-measure"]        += metrics["F-measure"]
            total_metrics["CMLc"]             += metrics["Correct Metric Level Continuous"]
            total_metrics["CMLt"]             += metrics["Correct Metric Level Total"]
            total_metrics["AMLc"]             += metrics["Any Metric Level Continuous"]
            total_metrics["AMLt"]             += metrics["Any Metric Level Total"]
            total_metrics["Information gain"] += metrics["Information gain"]
            valid_files += 1

    if valid_files > 0:
        print(f'\n--- Final Results (averaged over {valid_files} files) ---')
        avg = {k: v / valid_files for k, v in total_metrics.items()}
        header = f"{'F-measure':<10} | {'CMLc':<6} | {'CMLt':<6} | {'AMLc':<6} | {'AMLt':<6} | {'Info Gain':<10}"
        print(header)
        print('-' * 65)
        row = (
            f"{avg['F-measure']:<10.3f} | "
            f"{avg['CMLc']:<6.3f} | {avg['CMLt']:<6.3f} | "
            f"{avg['AMLc']:<6.3f} | {avg['AMLt']:<6.3f} | "
            f"{avg['Information gain']:<10.3f}"
        )
        print(row)
    else:
        print('No valid files evaluated.')

## 6. Visualisation

Plots the model's beat activation over a chosen time window, overlaid with
ground-truth beats (green solid) and predicted beats (red dashed). Useful for
qualitative debugging — does the activation actually peak where it should?

In [ ]:
def visualize_beat_tracking(track_id, ballroom_data, start_sec=5, end_sec=15,
                            model_path="tcn_beat_tracker.pth"):
    """Plot model activation, ground-truth beats, and predicted beats."""
    track = ballroom_data.track(track_id)
    sr = 44100
    hop_length = int(sr * 0.01)

    ref_beats = track.beats.times if track.beats is not None else np.array([])

    features = compute_log_cqt(track.audio_path)
    model = TCNBeatTracker()
    model.load_state_dict(
        torch.load(model_path, map_location=torch.device("cpu"), weights_only=True)
    )
    model.eval()
    with torch.no_grad():
        logits = model(features).squeeze()
        activations = torch.sigmoid(logits).cpu().numpy()

    est_beats, _ = beatTracker(track.audio_path)
    times = librosa.frames_to_time(np.arange(len(activations)), sr=sr, hop_length=hop_length)

    mask_ref = (ref_beats >= start_sec) & (ref_beats <= end_sec)
    mask_est = (np.array(est_beats) >= start_sec) & (np.array(est_beats) <= end_sec)
    window_ref = ref_beats[mask_ref]
    window_est = np.array(est_beats)[mask_est]

    plt.figure(figsize=(12, 6))
    plt.plot(times, activations, label="TCN activation (probability)", color="blue", alpha=0.7)

    for i, b in enumerate(window_ref):
        plt.axvline(x=b, color="green", linestyle="-", linewidth=2, alpha=0.8,
                    label="Ground truth" if i == 0 else "")
    for i, b in enumerate(window_est):
        plt.axvline(x=b, color="red", linestyle="--", linewidth=2, alpha=0.8,
                    label="Predicted beat" if i == 0 else "")

    plt.xlim([start_sec, end_sec])
    plt.ylim([0, 1.1])
    plt.title(f"Beat tracking — Track {track_id}")
    plt.xlabel("Time (s)")
    plt.ylabel("Beat probability")
    plt.legend(loc="upper right")
    plt.tight_layout()
    plt.show()

## 7. End-to-end pipeline

Run this cell to reproduce the full experiment: download Ballroom via
`mirdata`, pre-compute features, train the TCN, evaluate on the held-out 20%
test split, and plot a sample track.

**Note:** the full run takes ~30 mins for pre-computation plus ~30 mins for
training on CPU. To skip training and just evaluate the pre-trained model,
delete the `train_model` call and load `tcn_beat_tracker.pth` directly.

In [ ]:
if __name__ == "__main__":
    ballroom_init = mirdata.initialize("ballroom")
    ballroom_init.download()

    precompute_dataset(ballroom_init)

    all_track_ids = list(ballroom_init.track_ids)
    random.seed(42)
    random.shuffle(all_track_ids)
    split_idx = int(0.8 * len(all_track_ids))
    train_ids = all_track_ids[:split_idx]
    test_ids = all_track_ids[split_idx:]

    print(f"Total: {len(all_track_ids)}  |  Train: {len(train_ids)}  |  Test: {len(test_ids)}")

    train_dataset = PrecomputedBallroomDataset(track_ids=train_ids)
    train_loader = DataLoader(train_dataset, batch_size=1, shuffle=True)

    my_model = TCNBeatTracker()
    trained_model = train_model(my_model, train_loader, num_epochs=50)
    torch.save(trained_model.state_dict(), "tcn_beat_tracker.pth")

    evaluate_dataset(test_ids, ballroom_init)

    sample_track_id = test_ids[0]
    visualize_beat_tracking(sample_track_id, ballroom_init, start_sec=10, end_sec=20)